In [ ]:
!pip install hpsv3

In [ ]:
from hpsv3 import HPSv3RewardInferencer

inferencer = HPSv3RewardInferencer(device='cuda')

Flash Attention is not installed. Falling to SDPA.


`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen2VLRewardModelBT were not initialized from the model checkpoint at Qwen/Qwen2-VL-7B-Instruct and are newly initialized: ['rm_head.0.bias', 'rm_head.0.weight', 'rm_head.3.bias', 'rm_head.3.weight', 'rm_head.5.bias', 'rm_head.5.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import datasets
ds = datasets.load_dataset("weathon/critical_comparsion", split="train")

In [ ]:
import torch
def rate_hpsv3(sample):
  image_paths = [sample["DanceFlux"], sample["Flux Krea"]]
  prompts = [sample["prompt"]] * 2
  with torch.no_grad():
    rewards = inferencer.reward(prompts=prompts, image_paths=image_paths)
    scores = [reward[0].item() for reward in rewards]
  return {
      "DanceFlux": scores[0],
      "Flux Krea": scores[1],
  }

In [ ]:
datasets.config.IN_MEMORY_MAX_SIZE = 3_000_000_000

In [ ]:
hpsv3 = []
for sample in ds:
  hpsv3.append(rate_hpsv3(sample))

In [ ]:
ds = ds.add_column("hpsv3", hpsv3)

In [ ]:
ds.save_to_disk("critical_comparsion")

Saving the dataset (0/1 shards):   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_from_disk
ds = load_from_disk("critical_comparsion")

In [ ]:
import torch
import hpsv2

def rate_hpsv2(sample):
  image_paths = [sample["DanceFlux"], sample["Flux Krea"]]
  prompt = sample["prompt"]
  with torch.no_grad():
    scores = hpsv2.score(image_paths, prompt, hps_version="v2.1")

  return {
      "DanceFlux": scores[0],
      "Flux Krea": scores[1],
  }
hpsv2_scores = []
for sample in ds:
  hpsv2_scores.append(rate_hpsv2(sample))

/usr/local/lib/python3.12/dist-packages/hpsv2/img_score.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [ ]:
ds = ds.add_column("hpsv2", hpsv2_scores)


In [ ]:
ds.save_to_disk("critical_comparsion_2")

Saving the dataset (0/1 shards):   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
import ImageReward as RM
model = RM.load("ImageReward-v1.0")

ImageReward.pt:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

load checkpoint from /root/.cache/ImageReward/ImageReward.pt


med_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

checkpoint loaded


In [ ]:
import torch
import hpsv2

def rate_image_reward(sample):
  image_paths = [sample["DanceFlux"], sample["Flux Krea"]]
  prompt = sample["prompt"]
  with torch.no_grad():
    scores = model.score(prompt, image_paths)

  return {
      "DanceFlux": scores[0],
      "Flux Krea": scores[1],
  }
image_reward_scores = []
for sample in ds:
  image_reward_scores.append(rate_image_reward(sample))
ds = ds.add_column("image_reward", image_reward_scores)

In [ ]:
ds.push_to_hub("weathon/critical_comparsion")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/weathon/critical_comparsion/commit/d3fe5f37ff2eef79f2513255054795555fea8915', commit_message='Upload dataset', commit_description='', oid='d3fe5f37ff2eef79f2513255054795555fea8915', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weathon/critical_comparsion', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weathon/critical_comparsion'), pr_revision=None, pr_num=None)